In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

# PDF loading
from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader

# Text Splitting 
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeding
from langchain_openai import OpenAIEmbeddings

/var/folders/m8/jv_qd8md6_n84v8zcshwvjm40000gn/T/ipykernel_22947/828698334.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader
/Users/manojkrishnappa/github/FDE-AIML-COMPLETE-COURSE/.FDE-AIML-COMPLETE-COURSE/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1027 20:41:20.206000 22947 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
dir_loader = DirectoryLoader(
    "pdfdoc",
    glob = "*.pdf",
    loader_cls=PyMuPDFLoader
)

In [3]:
documents = dir_loader.load()
print(f"Total documents loaded is {len(documents)}")

Total documents loaded is 265


In [4]:
# Document Splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
)

chunks = text_splitter.split_documents(documents)
print(f"created chunks is {len(chunks)} from documents loaded is {len(documents)}")

created chunks is 729 from documents loaded is 265


In [5]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [6]:
## https://developers.openai.com/api/docs/guides/embeddings

embedding = OpenAIEmbeddings(
    model = "text-embedding-3-small"
)

In [7]:
sample_text = "Hello Everyone how are yoy...."
vector = embedding.embed_query(sample_text)

In [8]:
vector[:10]

[0.08502197265625,
 -0.040863037109375,
 -0.06689453125,
 0.042022705078125,
 0.046173095703125,
 0.01291656494140625,
 0.005687713623046875,
 -0.0047760009765625,
 -0.0180816650390625,
 -0.05322265625]

# ASTRA DB

https://docs.datastax.com/en/astra-db-serverless/index.html

In [9]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)


os.environ["ASTRA_DB_API_ENDPOINT"] = os.getenv("ASTRA_DB_API_ENDPOINT")
os.environ["ASTRA_DB_APPLICATION_TOKEN"] = os.getenv("ASTRA_DB_APPLICATION_TOKEN")

ASTRA_DB_API_ENDPOINT = os.getenv("ASTRA_DB_API_ENDPOINT")
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")

In [10]:
from langchain_astradb import AstraDBVectorStore
from astrapy.api_options import APIOptions,TimeoutOptions

api_options = APIOptions(
    timeout_options = TimeoutOptions(
        request_timeout_ms=180000,
        general_method_timeout_ms=180000
    )
)


vector_db = AstraDBVectorStore(
    embedding=embedding,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
    collection_name="astradb_vetcor_collections",
    namespace=None,
    api_options = api_options
)

In [11]:
vector_db.clear()

In [14]:
import hashlib
ids = [
    hashlib.md5(
        f"{c.metadata.get('source')}|{c.metadata.get('page')}|{c.page_content}".encode()
    ).hexdigest()
    for c in chunks
]


BATCH = 100
for i in range(0, len(chunks), BATCH):
    vector_db.add_documents(chunks[i:i + BATCH], ids=ids[i:i + BATCH])
    print(f"uploaded {min(i + BATCH, len(chunks))}/{len(chunks)}")


uploaded 100/729
uploaded 200/729
uploaded 300/729
uploaded 400/729
uploaded 500/729
uploaded 600/729
uploaded 700/729
uploaded 729/729


In [15]:
query = "what is Encoder.?"

similar_docs = vector_db.similarity_search(
    query,
    k=3
)
similar_docs

[Document(id='8bb2e7f1d2e97e8491adcf14a20bf562', metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': 'pdfdoc/NIPS-2017-attention-is-all-you-need-Paper.pdf', 'file_path': 'pdfdoc/NIPS-2017-attention-is-all-you-need-Paper.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 1}, page_content='connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,\nrespectively.\n3.1\nEncoder and Decoder Stacks\nEncoder:\nThe encoder is composed of a stack of N = 6 identical layers. Each layer has two\nsub-layers. The ﬁrst is a multi-head self-attention mechanism, and the sec

In [ ]:
retriver = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [62]:
query = "Explain the encoder in the Transformer paper."
docs = retriver.invoke(query)

for doc in docs:
    print(doc.metadata.get("source"), doc.metadata.get("page"))
    print(doc.page_content[:500], "\n")

pdfdoc/NIPS-2017-attention-is-all-you-need-Paper.pdf 2
Figure 1: The Transformer - model architecture.
wise fully connected feed-forward network. We employ a residual connection [10] around each of
the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimension dmodel = 512.
Decoder:
The  

pdfdoc/NIPS-2017-attention-is-all-you-need-Paper.pdf 1
aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate
self-attention and discuss its advantages over models such as [14, 15] and [8].
3
Model Architecture
Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 29].
Here, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a se

In [52]:
for doc in docs:
    print(doc.page_content)

Agreement No. SI2.304597 
Hospital Data Project 
 
 
5
1.2 The Importance of Hospital Data 
 
Information on hospital infrastructure, activity, personnel and costs is of considerable 
importance at national level.  Of course, one of the principal reasons for this is that hospitals 
have always been and continue to be large consumers of health service resources.  In recent 
years, a number of additional factors have contributed to an increasing imperative to 
maximise the benefits of good hospital data.  Hospital data are now required to serve a variety 
of purposes including supporting activity monitoring, performance measurement, casemix-
based funding, service planning and epidemiological analysis. Within countries, hospital data 
are now widely used to analyse regional performance and to identify areas that may require 
action.  
 
Furthermore, it is increasingly evident that the availability of truly comparable hospital data at
coverage, other projects etc. before setting out their

In [63]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model = "gpt-4o"
)

In [64]:
llm.invoke("Hello")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_bbf31a9d1e', 'id': 'chatcmpl-EddaFgUMKUAszR5hfZp5cy0emL8qz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17e05-9814-7f83-a3db-3d5d5488e9be-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [65]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

cutom_prompt = ChatPromptTemplate.from_template(
    """ 
    you are helpfull assistant.

    answer the question using only provided context

    if the answer cannot be found in context or the retriver result then say
    "I dont know your question please ask only in the knowlede base" 

    Context:
    {context}

    Question:
    {question}

    Answer:

    """
)

In [66]:
def formated_docs(docs):
    context = ""

    for doc in docs:
        context +=doc.page_content + "\n\n"

    return context

In [67]:
context = formated_docs(docs)

In [68]:
ragchain = (
    
    {
        "context":retriver | formated_docs,
        "question":RunnablePassthrough()
    }
    | cutom_prompt
    | llm
    | StrOutputParser()

)

In [69]:
query = "what is Encoder.?"

answer = ragchain.invoke(query)

print(answer)

The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers: the first is a multi-head self-attention mechanism, and the second is a position-wise fully connected feed-forward network. Residual connections are employed around each of the two sub-layers, followed by layer normalization. All sub-layers and embedding layers produce outputs of dimension d_model = 512. The encoder layers allow each position to attend to all positions in the previous layer of the encoder.


In [71]:
query = "Explain the encoder in the Transformer paper."

answer = ragchain.invoke(query)

print(answer)

The encoder in the Transformer model is composed of a stack of N = 6 identical layers. Each layer consists of two sub-layers. The first sub-layer is a multi-head self-attention mechanism, and the second is a simple, position-wise fully connected feed-forward network. A residual connection is employed around each of the two sub-layers, followed by layer normalization. This means the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. In order to facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension dmodel = 512.


In [72]:
query = "The Importance of Hospital Data"

answer = ragchain.invoke(query)

print(answer)

Hospital data is important because hospitals are major consumers of health service resources. Additionally, there is an increasing need to optimize the benefits of good hospital data, which is used for several purposes such as activity monitoring, performance measurement, casemix-based funding, service planning, and epidemiological analysis. Hospital data is also pivotal in analyzing regional performance within countries and identifying areas that may require intervention.


In [73]:
query = "what is Hospital Data Project (HDP) Objectives "

answer = ragchain.invoke(query)

print(answer)

The Hospital Data Project (HDP) had two key objectives. The first was the preparation of a detailed and practical methodology for the collection of comparable hospital activity data across Europe. The second was the production of a pilot data set according to the agreed methodology.
